In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('cleaned_data.csv')
df

,title,authors,average_rating,language_code,publisher,tags
0,Harry Potter and the Half Blood Prince Harry P...,J.K.Rowling/MaryGrandPré,4.57,eng,ScholasticInc.,harry potter and the half blood prince harry p...
1,Harry Potter and the Order of the Phoenix Harr...,J.K.Rowling/MaryGrandPré,4.49,eng,ScholasticInc.,harry potter and the order of the phoenix harr...
2,Harry Potter and the Chamber of Secrets Harry ...,J.K.Rowling,4.42,eng,Scholastic,harry potter and the chamber of secrets harry ...
3,Harry Potter and the Prisoner of Azkaban Harry...,J.K.Rowling/MaryGrandPré,4.56,eng,ScholasticInc.,harry potter and the prisoner of azkaban harry...
4,Harry Potter Boxed Set Books 1 5 Harry Potter ...,J.K.Rowling/MaryGrandPré,4.78,eng,Scholastic,harry potter boxed set books 1 5 harry potter ...
...,...,...,...,...,...,...
11072,Expelled from Eden A William T Vollmann Reader,WilliamT.Vollmann/LarryMcCaffery/MichaelHemmin...,4.06,eng,DaCapoPress,expelled from eden a william t vollmann reader...
11073,You Bright and Risen Angels,WilliamT.Vollmann,4.08,eng,PenguinBooks,"you bright and risen angels,williamt.vollmann,..."
11074,The Ice Shirt Seven Dreams 1,WilliamT.Vollmann,3.96,eng,PenguinBooks,"the ice shirt seven dreams 1 ,williamt.vollman..."
11075,Poor People,WilliamT.Vollmann,3.72,eng,Ecco,"poor people,williamt.vollmann,ecco"


In [3]:
df.info()
df.dropna(inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11077 entries, 0 to 11076
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           11077 non-null  object 
 1   authors         11077 non-null  object 
 2   average_rating  11077 non-null  float64
 3   language_code   11077 non-null  object 
 4   publisher       11077 non-null  object 
 5   tags            11077 non-null  object 
dtypes: float64(1), object(5)
memory usage: 519.4+ KB


In [4]:
df=df[df['title'].str.strip()!='']
df.reset_index(drop=True,inplace=True)

In [5]:
#let's find unique words
from sklearn.feature_extraction.text import CountVectorizer
temp_cv=CountVectorizer(stop_words='english')
temp_cv.fit(df['tags'])
print('Total uniqye words are=',len(temp_cv.vocabulary_))

Total uniqye words are= 22725


In [6]:
#lets perform vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=5000,stop_words='english')
vector=tfidf.fit_transform(df['tags']).toarray()

In [7]:
vector

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(11077, 5000))

In [8]:
#cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
similarity=cosine_similarity(vector)
similarity

array([[1.        , 0.79159028, 0.69640783, ..., 0.        , 0.        ,
        0.        ],
       [0.79159028, 1.        , 0.71367006, ..., 0.        , 0.        ,
        0.        ],
       [0.69640783, 0.71367006, 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.42815006,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.42815006, 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        1.        ]], shape=(11077, 11077))

In [9]:
#now let's test small prediction
def get_name_by_index(i):
    if i<len(df) and i>=0:
        return df.loc[i,'title']
    else:
        return ''

In [19]:
a=get_name_by_index(12)
a

'A Short History of Nearly Everything'

In [11]:
#one more to take book name as an input 
def get_index_from_name(name):
    clean_user_name = name.strip().lower().replace(' ', '').replace('-', '')
    match = df[df['title'].str.lower().str.replace(' ', '').str.replace('-', '') == clean_user_name]
    if not match.empty:
        return match.index[0]
    return -1

In [12]:
b=get_index_from_name('Unauthorized Harry Potter Book Seven News  Half Blood Prince Analysis and Speculation')
print(b)

5


In [20]:
#let's predict for user input
name = input("Enter book name that you read:")
index = get_index_from_name(name)
if index != -1:
    similarity_indexes =  list(enumerate(similarity[index]))
    similarity_indexes = sorted(similarity_indexes, key=lambda x: x[1], reverse=True)
    # for i in range(1,6):
    #     idx=similarity_indexes[i][0]
    #     print(idx,repr(df.loc[idx,'title']))
              
    print("You must read following books:")
    for i in range(1, 6):
        print(i, ":", get_name_by_index(similarity_indexes[i][0]))
else:
    print("Book not Found") 

Enter book name that you read: A Short History of Nearly Everything


You must read following books:
1 : A Short History of Nearly Everything Illustrated Edition 
2 : In a Sunburned Country
3 : Thumbsucker
4 : Bill Bryson s African Diary
5 : The Life and Times of the Thunderbolt Kid A Memoir


In [21]:
#now let's export the similarity
import pickle as pkl
pkl.dump(similarity,open('similarity.pkl','wb+'))